# Deliverable 4 – INF201

**Group members:** _[Mathias Gundersen and Jakub Mermer]_
**GitHub repository:** _[https://github.com/saltonese/Del_4-G6_5.git]_

This notebook contains

1. **Task 1 – Temperatures**: the `Temperature` class, a demonstration, and the `pytest` output
2. **Task 2 – Fractions**: the `Fraction` class, a demonstration, and the `pytest` output
3. **Discussion of GenAI use**

The classes live in `Task_1_temperature.py` and `Task_2_fraction.py`, and the tests in `Temperature_test.py` and
`Fraction_test.py`. The notebook imports the classes from those files and runs the tests with `pytest`.

In [22]:
import sys
import pytest
from pathlib import Path
from IPython.display import Code

project_dir = Path.cwd()
if not (project_dir / "Scripts").is_dir():
    project_dir = project_dir.parent

sys.path.insert(0, str(project_dir / "Scripts"))

print("Python:", sys.version.split()[0])
print("pytest:", pytest.__version__)
print(Path.cwd())


Python: 3.13.14
pytest: 9.2.0
c:\Users\mathi\StudieNMBU\PROG\Del_4-G6_5\Notebooks


---
# Task 1: Temperatures

## Design choices BESKRIV valgene du tar


- **Case-insensitive units.** The unit is normalised to upper case, so `"c"` and `"C"` give the same object.


## The code

## Demonstraion

---
# Task 2: Fractions

## Design choices

- **Validation.** Numerator and denominator must be integers (`TypeError` otherwise; `bool` is rejected explicitly,
  because it is a subclass of `int` in Python) and the denominator must be strictly positive (`ValueError`).
- **Lowest terms.** Fractions are reduced with `math.gcd` in the constructor, so `Fraction(2, 4)` is stored as `1/2`.
  This keeps equality and hashing simple.
- **Comparisons use cross-multiplication**, with no floating point: `a/b < c/d` is the same as `a*d < c*b`.
  This is only correct when both denominators are positive (multiplying an inequality by a negative number flips it),
  which is exactly why the task requires `denominator > 0`. The sign of a fraction is therefore always kept in the numerator.
- **Special methods implemented:** `__init__`, all six comparisons, and in addition `__repr__`, `__str__`, `__add__`,
  `__sub__`, `__mul__`, `__truediv__`, `__neg__`, `__abs__`, `__float__` and `__hash__` (the task requires at least four).
- **Operations with other types** return `NotImplemented`, so Python raises `TypeError`.
- **Hashable.** Unlike `Temperature`, this is safe, because equality is exact and fractions are in lowest terms.
  Equal fractions therefore always have the same `(numerator, denominator)` and the same hash.

## The code

In [16]:

Code(filename=r"..\Scripts\Task_2_fraction.py", language="python")

"""A Fraction class with comparionsn and arithmatic via special methods """

from math import gcd #greatest commom denominator


class Fraction:
    """An imutable fraction numerator/denominatior
    
    The denominatior must be a strictly positive integer. Fractions are stored
    lowest terms, so Fraction(2, 4) and Fraction(1, 2) hold identical data.
    """

    def __init__(self, numerator, denominator = 1):
        for name, part in (("numarator", numerator),("denominator", denominator)):
            # bool is a subclass of int in pyhton, so we exclude it
            if isinstance(part, bool) or not isinstance(part, int):
                raise TypeError(f"{name} must be an integer, got {type(part).__name__}")
        if denominator <= 0:
            raise ValueError("denominator must be more than 0 (X > 0)")

        # reduce to lowest terms. gcd is always >= 1 because denominator > 0
        divisor = gcd(numerator, denominator)
        self._numerator = numerator // divisor
        self._denominator = denominator // divisor

    #----------------------------------------------------------------------------
    # Read-only attributes
    #--------------------------------------------------------------------------------
    @property
    def numerator(self):
        return self._numerator

    @property
    def denominator(self):
        return self._denominator

    #-----------------------------------------------------------------------------
    # String repersentaions
    #----------------------------------------------------------------------
    def __repr__(self):
        return f"Fraction({self._numerator}, {self._denominator})"

    def __str__(self):
        if self._denominator == 1:
            return str(self._numerator)
        return f"{self._numerator}/{self._denominator}"

    #---------------------------------------------------------------------
    # Comparisons. Cross-multiplication avoids floating point entierly:
    #   a/b < c/d  <=>  a*b < c*d   (valid beacuse b, d > 0)
    # This is why the denominator must be positive: multipying an inequality
    # with a negative number would flip its dirction
    # -----------------------------------------------------------
    def __eq__(self, other):
        if not isinstance(other, Fraction):
            return NotImplemented
        return self._numerator * other._denominator == other._numerator * self._denominator

    def __ne__(self, other):
        result = self.__eq__(other)
        return result if result is NotImplemented else not result

    def __lt__(self, other):
        if not isinstance(other, Fraction):
            return NotImplemented
        return self._numerator * other._denominator < other._numerator * self._denominator

    def __le__ (self, other):
        if not isinstance(other, Fraction):
            return NotImplemented
        return self._numerator * other._denominator <= other._numerator * self._denominator

    def __gt__(self, other):
        if not isinstance(other, Fraction):
            return NotImplemented
        return self._numerator * other._denominator > other._numerator * self._denominator

    def __ge__(self, other):
        if not isinstance(other, Fraction):
            return NotImplemented
        return self._numerator * other._denominator >= other._numerator * self._denominator

    #--------------------------------------------------------------------
    # arithmetic
    #---------------------------------------------------------------------
    def __add__(self, other):
        if not isinstance (other, Fraction):
            return NotImplemented
        return Fraction(
            self._numerator * other._denominator + other._numerator * self._denominator,
            self._denominator * other._denominator,
        )

    def __sub__(self, other):
        if not isinstance(other, Fraction):
            return NotImplemented
        return Fraction(
            self._numerator * other._denominator - other

## Demonstration

In [17]:
from Task_2_fraction import Fraction

a, b = Fraction(1, 2), Fraction(1, 3)
print("repr:", repr(a), "| str:", a)
print("Reduced to lowest terms:", repr(Fraction(6, 8)))
print("Whole number prints without denominator:", Fraction(4, 2))
print()

print("Arithmetic with a =", a, "and b =", b)
print("  a + b =", a + b)
print("  a - b =", a - b)
print("  a * b =", a * b)
print("  a / b =", a / b)
print("  -a    =", -a, "| abs(-3/4) =", abs(Fraction(-3, 4)))
print("  float(a) =", float(a))
print("  1/2 / (-1/4) =", Fraction(1, 2) / Fraction(-1, 4), "(sign stays in the numerator)")

repr: Fraction(1, 2) | str: 1/2
Reduced to lowest terms: Fraction(3, 4)
Whole number prints without denominator: 2

Arithmetic with a = 1/2 and b = 1/3
  a + b = 5/6
  a - b = 1/6
  a * b = 1/6
  a / b = 3/2
  -a    = -1/2 | abs(-3/4) = 3/4
  float(a) = 0.5
  1/2 / (-1/4) = -2 (sign stays in the numerator)


In [18]:
print("Comparisons with a =", a, "and b =", b)
print("  a == b:", a == b, "| a == 2/4:", a == Fraction(2, 4))
print("  a != b:", a != b)
print("  a <  b:", a < b, "| a <= b:", a <= b)
print("  a >  b:", a > b, "| a >= b:", a >= b)
print()

print("Sorting:", [str(f) for f in sorted([Fraction(3, 4), Fraction(-1, 2), Fraction(1, 3), Fraction(1, 2)])])
print("Equal fractions in a set:", {Fraction(1, 2), Fraction(2, 4), Fraction(1, 3)})

Comparisons with a = 1/2 and b = 1/3
  a == b: False | a == 2/4: True
  a != b: True
  a <  b: False | a <= b: False
  a >  b: True | a >= b: True

Sorting: ['-1/2', '1/3', '1/2', '3/4']
Equal fractions in a set: {Fraction(1, 2), Fraction(1, 3)}


In [19]:
print("Invalid input and operations:")
for args in [(1, 0), (1, -2)]:
    try:
        Fraction(*args)
    except ValueError as error:
        print(f"  Fraction{args} -> ValueError: {error}")
for args in [(1.5, 2), ("1", 2)]:
    try:
        Fraction(*args)
    except TypeError as error:
        print(f"  Fraction{args} -> TypeError: {error}")
try:
    Fraction(1, 2) / Fraction(0, 5)
except ZeroDivisionError as error:
    print("  1/2 / 0 -> ZeroDivisionError:", error)
try:
    Fraction(1, 2) + 1
except TypeError as error:
    print("  Fraction(1, 2) + 1 -> TypeError:", error)

Invalid input and operations:
  Fraction(1, 0) -> ValueError: denominator must be more than 0 (X > 0)
  Fraction(1, -2) -> ValueError: denominator must be more than 0 (X > 0)
  Fraction(1.5, 2) -> TypeError: numarator must be an integer, got float
  Fraction('1', 2) -> TypeError: numarator must be an integer, got str
  1/2 / 0 -> ZeroDivisionError: divison by a fraction equal to 0
  Fraction(1, 2) + 1 -> TypeError: unsupported operand type(s) for +: 'Fraction' and 'int'


## Tests

`test_fraction.py` is organised as follows:

| Test class | What it checks |
|---|---|
| `TestInitialiser` | stored values, default denominator, reduction, negative and zero numerators, invalid denominators (`ValueError`), non-integers (`TypeError`), read-only attributes |
| `TestComparisons` | `==`, `!=`, `<`, `<=`, `>`, `>=` including negative fractions and different denominators, `NotImplemented` for other types, sorting |
| `TestStringRepresentations` | `__repr__` (including `eval(repr(f)) == f`) and `__str__` |
| `TestArithmetic` | `+`, `-`, `*`, `/` (also division by a negative fraction and by zero), `-f`, `abs(f)`, no mutation of the operands, `TypeError` for other types |
| `TestConversionAndHashing` | `float(f)`, equal hashes for equal fractions, use in sets and as dictionary keys |

In [24]:
!{sys.executable} -m pytest "..\Scripts\Fraction_test.py" -v --no-header -p no:cacheprovider

============================= test session starts =============================
collecting ... collected 46 items

..\Scripts\Fraction_test.py::TestInitialiser::test_stores_numerator_and_denominator PASSED [  2%]
..\Scripts\Fraction_test.py::TestInitialiser::test_default_denominator_is_one PASSED [  4%]
..\Scripts\Fraction_test.py::TestInitialiser::test_is_reduced_to_lowest_terms PASSED [  6%]
..\Scripts\Fraction_test.py::TestInitialiser::test_negative_numerator_is_allowed PASSED [  8%]
..\Scripts\Fraction_test.py::TestInitialiser::test_zero_numerator PASSED [ 10%]
..\Scripts\Fraction_test.py::TestInitialiser::test_non_positive_denominator_raises_value_error[0] PASSED [ 13%]
..\Scripts\Fraction_test.py::TestInitialiser::test_non_positive_denominator_raises_value_error[-1] PASSED [ 15%]
..\Scripts\Fraction_test.py::TestInitialiser::test_non_positive_denominator_raises_value_error[-5] PASSED [ 17%]
..\Scripts\Fraction_test.py::TestInitialiser::test_non_integers_raise_type_error[1.5-2] PA

## All test togehter

'tail' is not recognized as an internal or external command,
operable program or batch file.
